## Train SRU model with embeddings-resnet50-100

Hope it will work...

### Installing SRU with v3.0.0

In [1]:
!pip install sru==3.0.0.dev6

### Importing Necessary Libraries

In [2]:
import logging
import os

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sru import SRUpp
from torch.nn import CrossEntropyLoss
from torch.nn import Module, Dropout, Linear
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import gc

### Set the Device to CUDA

In [3]:
# Set the device to use (e.g., 'cpu', 'cuda', 'mps')
device = torch.device("mps" if torch.backends.mps.is_available() else
                      ("cuda" if torch.cuda.is_available() else "cpu"))

# Select Device According to Availability
print("Device selected:", device)

# If the device is CUDA, print the device capability
if device.type == "cuda":
    os.system("nvidia-smi")
    print()
    print("Device type:", device.type)
    print("Capability:", torch.cuda.get_device_capability(device))
else:
    print("Device capabilities are limited on MPSs and CPUs.")


Device selected: cuda
Mon Apr  8 10:26:54 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla P100-PCIE-16GB           Off | 00000000:00:04.0 Off |                    0 |
| N/A   34C    P0              26W / 250W |      2MiB / 16384MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+---

In [4]:
embedding_file = np.load('/kaggle/input/multiclass-embeddings-resnet/multiclass_train_embeddings.npy')
label_file = np.load('/kaggle/input/multiclass-embeddings-resnet/multiclass_train_labels.npy')

In [5]:
inference_sample_size = 0.05
test_sample_size = 0.2

In [6]:
# Inference split
embedding, inference_sample_embedding, labels, inference_sample_label = train_test_split(
    embedding_file, label_file, test_size=inference_sample_size, random_state=42
)

# Train Test Split
x_train, x_test, y_train, y_test = train_test_split(
    embedding, labels, test_size=test_sample_size, random_state=42
)

In [7]:
train_embeddings = torch.from_numpy(x_train).to(device)
train_labels = torch.from_numpy(y_train).to(device)
test_embeddings = torch.from_numpy(x_test).to(device)
test_labels = torch.from_numpy(y_test).to(device)
inference_embeddings = torch.from_numpy(inference_sample_embedding).to(device)
inference_labels = torch.from_numpy(inference_sample_label).to(device)

In [8]:
# Delete initial file variables
del embedding_file
del label_file

# Delete 2nd order variables (splitting)
del embedding
del labels
del inference_sample_embedding
del inference_sample_label
del x_train
del x_test
del y_train
del y_test

In [9]:
batch_size = 8

# Create TensorDataset
train_data = TensorDataset(train_embeddings, train_labels)
test_data = TensorDataset(test_embeddings, test_labels)
inference_data = TensorDataset(inference_embeddings, inference_labels)

train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_data, shuffle=True, batch_size=batch_size)
inference_loader = DataLoader(inference_data, shuffle=True, batch_size=batch_size)

### Train Test Split, Convert to torch.tensor and View Shape

In [10]:
class SRUppModel(Module):
    def __init__(self, input_size, hidden_size, **kwargs):
        super(SRUppModel, self).__init__()
        # Main SRU layer
        self.sru_layers = SRUpp(
            proj_size = 2048,
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=kwargs.get('num_layers', 2),
            dropout=kwargs.get('dropout_prob', 0.0),
            bidirectional=kwargs.get('bidirectional', False),
            layer_norm=kwargs.get('layer_norm', True),
            highway_bias=kwargs.get('highway_bias', 0.0),
            rescale=kwargs.get('rescale', True),
            nn_rnn_compatible_return=kwargs.get('nn_rnn_compatible_return', False),
            proj_input_to_hidden_first=kwargs.get('proj_input_to_hidden_first', False),
            normalize_after=kwargs.get('normalize_after', False),
        ).to(device)
        # Dropout layer
        self.dropout = Dropout(kwargs.get('dropout_layer_prob', 0.2)).to(device)
        # Linear layer (Fully connected layer)
        self.linear = Linear(
            hidden_size, 
            out_features=kwargs.get('num_classes', 12)
        ).to(device)
        # L2 regularization
        self.l2_reg_lambda = kwargs.get('l2_reg_lambda', 1e-5)

    def forward(self, x):
        output_states, _, _ = self.sru_layers(x)
        output = self.linear(self.dropout(output_states[-1]))
        return output, output_states

    def l2_regularization(self):
        l2_reg = torch.tensor(0., device=device)
        for param in self.parameters():
            l2_reg += torch.norm(param, p=2)
        return self.l2_reg_lambda * l2_reg


model = SRUppModel(2048, 1024)
model = model.to(device)
model

SRUppModel(
  (sru_layers): SRUpp(
    (rnn_lst): ModuleList(
      (0): SRUppCell(2048, 1024, rescale=True, layer_norm=True, normalize_after=False,
        transform_module=SRUppAttention(
        (dropout): Dropout(p=0.0, inplace=False)
        (linear1): Linear(in_features=2048, out_features=2048, bias=False)
        (linear2): Linear(in_features=2048, out_features=4096, bias=False)
        (linear3): Linear(in_features=2048, out_features=4096, bias=False)
        (layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      )
      )
      (1): SRUppCell(1024, 1024, rescale=True, layer_norm=True, normalize_after=False,
        transform_module=SRUppAttention(
        (dropout): Dropout(p=0.0, inplace=False)
        (linear1): Linear(in_features=1024, out_features=2048, bias=False)
        (linear2): Linear(in_features=2048, out_features=4096, bias=False)
        (linear3): Linear(in_features=2048, out_features=3072, bias=False)
        (layer_norm): LayerNorm((2048,), 

In [11]:
# Define your loss function and optimizer
criterion = CrossEntropyLoss()
optimizer = Adam(model.parameters())

In [12]:
class CreateCluster:
    def __init__(self):
        # Define Clusters
        self.normal = []
        self.arson = []
        self.abuse = []
        self.assault = []
        self.burglary = []
        self.explosion = []
        self.fighting = []
        self.roadAccident = []
        self.robbery = []
        self.shooting = []
        self.stealing = []
        self.vandalism = []

        # Define Centers
        self.centers = []

    def add(self, embedding, label: int):
        if label == 0:
            self.normal.append(embedding)
        elif label == 2:
            self.arson.append(embedding)
        elif label == 1:
            self.abuse.append(embedding)
        elif label == 3:
            self.assault.append(embedding)
        elif label == 5:
            self.burglary.append(embedding)
        elif label == 6:
            self.explosion.append(embedding)
        elif label == 7:
            self.fighting.append(embedding)
        elif label == 4:
            self.roadAccident.append(embedding)
        elif label == 8:
            self.robbery.append(embedding)
        elif label == 9:
            self.shooting.append(embedding)
        elif label == 10:
            self.stealing.append(embedding)
        elif label == 11:
            self.vandalism.append(embedding)
        
        
    def __repr__(self):
        return f'Normal: {len(self.normal)}, Arson: {len(self.arson)}, Abuse: {len(self.abuse)}, Assault: {len(self.assault)}, Burglary: {len(self.burglary)}, Explosion: {len(self.explosion)}, Fighting: {len(self.fighting)}, Road Accident: {len(self.roadAccident)}, Robbery: {len(self.robbery)}, Shooting: {len(self.shooting)}, Stealing: {len(self.stealing)}, Vandalism: {len(self.vandalism)}'
    

In [13]:
# create cluster

cluster = CreateCluster()

model.train()
epochs = 100

total_correct_all_epochs = 0
total_samples_all_epochs = 0

# Setup logging
logging.basicConfig(filename='training.log', level=logging.INFO)

for epoch in range(epochs):
    total_correct = 0
    total_samples = 0
    total_loss = 0.0

    # Create tqdm progress bar for training loader
    progress_bar = tqdm(enumerate(train_loader), desc=f"Epoch {epoch + 1}/{epochs}", total=len(train_loader))

    for i, (videos, labels) in progress_bar:
#         print(videos.shape)  #torch.Size([8, 150, 2048])
        videos = videos.permute(1, 0, 2)
        # Forward pass
        outputs, temporal_features = model(videos)
#         print(temporal_features.shape) #torch.Size([150, 8, 1024])
        labels = labels.long()  # Convert labels to Long type
                
        # Put output into cluster with label when epoch = epochs
        if epoch == 99:
            temporal_features = temporal_features.permute(1, 0, 2)
#             print(temporal_features.shape) #torch.Size([8, 150, 1024])

            for temp_f, label in zip(temporal_features, labels):
                l = label.item()
                # Add embedding and label to cluster one by one
#                 print(temp_f.shape)
                temp_f = temp_f.reshape(temp_f.shape[0]*temp_f.shape[1])
#                 print(temp_f.shape)
                cluster.add(temp_f, l)        
        
        loss = criterion(outputs, labels) + model.l2_regularization()  # calculates loss
        total_loss += loss.item()
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # Calculate accuracy per batch
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        total_correct += (predicted == labels).sum().item()
        batch_accuracy = 100 * total_correct / total_samples

        # Update progress bar
        progress_bar.set_postfix(loss=loss.item(), accuracy=batch_accuracy)
        
        # Update overall accuracy counters
        total_correct_all_epochs += total_correct
        total_samples_all_epochs += total_samples
        
    # Log epoch statistics
    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = 100 * total_correct / total_samples
    logging.info(f'Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss}, Accuracy: {epoch_accuracy}%')

    print(f'Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss}, Accuracy: {epoch_accuracy}%')

# Close logging
logging.shutdown()

overall_accuracy = 100 * total_correct_all_epochs / total_samples_all_epochs
print(f'Average Train Accuracy of the model on the train videos: {overall_accuracy}%')

Epoch 1/100: 100%|██████████| 665/665 [00:38<00:00, 17.48it/s, accuracy=51.6, loss=2.97]


Epoch [1/100], Loss: 2.3752632903873474, Accuracy: 51.61654135338346%


Epoch 2/100: 100%|██████████| 665/665 [00:37<00:00, 17.96it/s, accuracy=64.6, loss=2.22]


Epoch [2/100], Loss: 1.1614645468897389, Accuracy: 64.58646616541354%


Epoch 3/100: 100%|██████████| 665/665 [00:37<00:00, 17.93it/s, accuracy=71.2, loss=0.28]


...

Epoch [98/100], Loss: 0.10669434475887539, Accuracy: 98.51503759398496%


Epoch 99/100: 100%|██████████| 665/665 [00:37<00:00, 17.81it/s, accuracy=98.8, loss=0.0191]


Epoch [99/100], Loss: 0.10382759154010984, Accuracy: 98.7593984962406%


Epoch 100/100: 100%|██████████| 665/665 [00:37<00:00, 17.57it/s, accuracy=98.5, loss=0.019]


Epoch [100/100], Loss: 0.11801677170048532, Accuracy: 98.47744360902256%
Average Train Accuracy of the model on the train videos: 94.83976946871684%


In [14]:
# print(cluster)
import torch

center = []

normal = [tensor.cpu().detach().numpy() for tensor in cluster.normal]
arson = [tensor.cpu().detach().numpy() for tensor in cluster.arson]
abuse = [tensor.cpu().detach().numpy() for tensor in cluster.abuse]
assault = [tensor.cpu().detach().numpy() for tensor in cluster.assault]
burglary = [tensor.cpu().detach().numpy() for tensor in cluster.burglary]
explosion = [tensor.cpu().detach().numpy() for tensor in cluster.explosion]
fighting = [tensor.cpu().detach().numpy() for tensor in cluster.fighting]
roadAccident = [tensor.cpu().detach().numpy() for tensor in cluster.roadAccident]
robbery = [tensor.cpu().detach().numpy() for tensor in cluster.robbery]
shooting = [tensor.cpu().detach().numpy() for tensor in cluster.shooting]
stealing = [tensor.cpu().detach().numpy() for tensor in cluster.stealing]
vandalism = [tensor.cpu().detach().numpy() for tensor in cluster.vandalism]

center.append(np.mean(normal, axis=0))
center.append(np.mean(arson, axis=0))
center.append(np.mean(abuse, axis=0))
center.append(np.mean(assault, axis=0))
center.append(np.mean(burglary, axis=0))
center.append(np.mean(explosion, axis=0))
center.append(np.mean(fighting, axis=0))
center.append(np.mean(roadAccident, axis=0))
center.append(np.mean(robbery, axis=0))
center.append(np.mean(shooting, axis=0))
center.append(np.mean(stealing, axis=0))
center.append(np.mean(vandalism, axis=0))

In [15]:

train_embedding = np.concatenate((normal, arson, abuse, assault, burglary, explosion, fighting, roadAccident, robbery, shooting, stealing, vandalism), axis=0)

train_labels = []

for i in range(len(normal)):
    train_labels.append(0)
    
for i in range(len(arson)):
    train_labels.append(2)
    
for i in range(len(abuse)):
    train_labels.append(1)
    
for i in range(len(assault)):
    train_labels.append(3)
    
for i in range(len(burglary)):
    train_labels.append(5)
    
for i in range(len(explosion)):
    train_labels.append(6)
    
for i in range(len(fighting)):
    train_labels.append(7)
    
for i in range(len(roadAccident)):
    train_labels.append(4)
    
for i in range(len(robbery)):
    train_labels.append(8)
    
for i in range(len(shooting)):
    train_labels.append(9)
    
for i in range(len(stealing)):
    train_labels.append(10)
    
for i in range(len(vandalism)):
    train_labels.append(11)
    
train_labels = np.array(train_labels)



len(train_embedding), len(train_labels)

(5320, 5320)

In [16]:
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler

# # Assuming centers and category_lists are already defined

# # Convert centers to numpy array
# centers_np = np.array([c for c in center if center is not None])

# # Define a list of labels for each category
# category_labels = ['Normal', 'Abuse', 'Arson', 'Assault', 'Road Accident', 'Burglary', 'Explosion', 'Fighting', 'Robbery', 'Shooting', 'Stealing', 'Vandalism']

# # Create a list to store embeddings for each category
# category_embeddings = []

# # Convert category lists to numpy arrays and store them in category_embeddings
# for label in range(len(category_labels)):
#     embeddings = np.array([emb.cpu().detach().numpy() for emb, l in zip(train_embeddings, train_labels) if l == label])
#     category_embeddings.append(embeddings)

# # Apply normalization to each category's embeddings
# normalized_embeddings = [StandardScaler().fit_transform(embeddings.reshape(embeddings.shape[0], -1)) for embeddings in category_embeddings]

# # Apply PCA for dimensionality reduction
# pca = PCA(n_components=2)
# embeddings_2d = [pca.fit_transform(embeddings) for embeddings in normalized_embeddings]

# # Create a list to store colors for each category
# colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'purple', 'orange', 'brown', 'pink', 'gray']

# # Plot clusters and centers
# plt.figure(figsize=(10, 8))
# for i, embeddings in enumerate(embeddings_2d):
#     plt.scatter(embeddings[:, 0], embeddings[:, 1], label=category_labels[i], alpha=0.5, color=colors[i])
# if len(centers_np) > 0:
#     plt.scatter(centers_np[:, 0], centers_np[:, 1], marker='x', color='red', s=100, label='Cluster Centers')
# plt.title('All Clusters with Centers')
# plt.xlabel('Principal Component 1')
# plt.ylabel('Principal Component 2')
# plt.legend()
# plt.grid(True)
# plt.show()


In [17]:
model.eval()

with torch.no_grad():
    correct = 0
    total = 0
    for videos, labels in test_loader:
        videos = videos.permute(1, 0, 2)
        outputs, t = model(videos)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        #print test accuracy per test

    print(f'Test Accuracy without cluster weight: {100 * correct / total}%')


Test Accuracy without cluster weight: 93.15789473684211%


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

category_names = ["Normal", "Abuse", "Arson", "Assault", "Road Accident", "Burglary", 
                  "Explosion", "Fighting", "Robbery", "Shooting", "Stealing", "Vandalism"]

def cosine_similarity_with_centers(embedding, center):
    similarities = [cosine_similarity([embedding], [c])[0][0] for c in center]
    return similarities

def predict_cluster(embedding, center):
    similarities = cosine_similarity_with_centers(embedding, center)
    max_similarity_index = np.argmax(similarities)
    return max_similarity_index

In [19]:
import torch.nn.functional as F

model.eval()

with torch.no_grad():
    correct = 0
    total = 0
    for videos, labels in test_loader:
        test_embeddings = []
        true_labels = []
        videos = videos.permute(1, 0, 2)
        outputs, t = model(videos)
#         _, predicted = torch.max(outputs.data, 1)
#         total += labels.size(0)
#         correct += (predicted == labels).sum().item()
        
        t = t.permute(1, 0, 2)
#         print(t.shape)
        for v, l in zip(t, labels):
            v = v.reshape(-1)
            l = l.item()
#             print(v.shape)
            test_embeddings.append(v)
            true_labels.append(l)
        
        test_embeddings = [tensor.cpu().detach().numpy() for tensor in test_embeddings]
        
        
        # List to store predicted cluster indices
        predicted_cluster_indices = []

        for emb, label in zip(test_embeddings, true_labels):
            predicted_cluster_index = predict_cluster(emb, center)
            predicted_cluster_indices.append(predicted_cluster_index)
        
    
        for i, index in enumerate(predicted_cluster_indices):
            outputs[i][index] += 0.5
        
#         normalized_outputs = F.softmax(outputs, dim=1)
        
        _, prediction = torch.max(outputs.data, 1)
        total += labels.size(0)
    
        correct += (prediction == labels).sum().item()

        
print(f'Test Accuracy with cluster weight: {100 * correct / total}%')


Test Accuracy with cluster weight: 93.15789473684211%
